# Prophet

In [ ]:
%pip install -q mlflow prophet scikit-learn pandas matplotlib

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    pass
sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from prophet import Prophet
from src.features import HOLIDAY_DATES
import logging
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

from src.data import load_raw, submission_ids
from src.features import WalmartFeatureBuilder, MARKDOWN_COLS
from src.metrics import wmae, holiday_weights
from src.validation import year_back_split, time_folds, DEV_TRAIN_END, TRAIN_END
from src.tracking import setup_mlflow, REGISTERED_MODEL_NAME

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("MLflow tracking:", setup_mlflow("Prophet_Training"))

In [ ]:
raw = load_raw()
train, test = raw["train"], raw["test"]

tr_idx, va_idx = year_back_split(train["Date"])
dev_train, dev_valid = train.iloc[tr_idx], train.iloc[va_idx]
print(f"train: {train.shape}, test: {test.shape}")
print(f"dev_train: {len(dev_train):,} რიგი (<= {DEV_TRAIN_END.date()}), "
      f"dev_valid: {len(dev_valid):,} რიგი (39 კვირა, შეიცავს Thanksgiving/Christmas/Super Bowl-ს)")

In [ ]:
hol_frames = []
for name, dates in HOLIDAY_DATES.items():
    lower = -14 if name in ("Thanksgiving", "Christmas") else -7
    hol_frames.append(pd.DataFrame({"holiday": name, "ds": dates,
                                    "lower_window": lower, "upper_window": 0}))
holidays_df = pd.concat(hol_frames, ignore_index=True)
holidays_df.head(8)

In [ ]:
def dept_shares(frame):
    tot = frame.groupby("Store")["Weekly_Sales"].sum()
    s = (frame.groupby(["Store", "Dept"])["Weekly_Sales"].sum()
         .reset_index(name="dept_sum"))
    s["share"] = s["dept_sum"] / s["Store"].map(tot)
    return s[["Store", "Dept", "share"]]

def disagg_predict(store_fc, shares, frame):
    m = (frame.merge(store_fc, on=["Store", "Date"], how="left")
              .merge(shares, on=["Store", "Dept"], how="left"))
    return (m["store_pred"] * m["share"]).fillna(0.0).to_numpy()

def fit_store_prophet(frame, horizon_dates, changepoint_prior_scale=0.05,
                      yearly_order=10, keep_model_for=None):
    preds, kept = [], None
    for store, g in frame.groupby("Store"):
        s = g.groupby("Date")["Weekly_Sales"].sum().reset_index()
        s.columns = ["ds", "y"]
        m = Prophet(yearly_seasonality=yearly_order, weekly_seasonality=False,
                    daily_seasonality=False, holidays=holidays_df,
                    changepoint_prior_scale=changepoint_prior_scale)
        m.fit(s)
        fc = m.predict(pd.DataFrame({"ds": horizon_dates}))
        preds.append(pd.DataFrame({"Store": store, "Date": horizon_dates,
                                   "store_pred": fc["yhat"].to_numpy()}))
        if store == keep_model_for:
            kept = (m, fc)
    return pd.concat(preds, ignore_index=True), kept

In [ ]:
valid_dates = pd.date_range(dev_valid["Date"].min(), dev_valid["Date"].max(), freq="W-FRI")

with mlflow.start_run(run_name="Prophet_StoreLevel_Disagg"):
    store_fc, kept = fit_store_prophet(dev_train, valid_dates, keep_model_for=1)
    shares = dept_shares(dev_train)
    pred = disagg_predict(store_fc, shares, dev_valid)
    prophet_wmae = wmae(dev_valid["Weekly_Sales"], pred, dev_valid["IsHoliday"])
    mlflow.log_params({"level": "store + dept-share disagg",
                       "yearly_fourier_order": 10, "changepoint_prior_scale": 0.05,
                       "holiday_windows": "TG/XMas -14d, SB/LD -7d"})
    mlflow.log_metric("valid_wmae", prophet_wmae)

    fig = kept[0].plot_components(kept[1])
    mlflow.log_figure(fig, "prophet_components_store1.png")
    plt.show()

print(f"Prophet disagg valid WMAE: {prophet_wmae:.0f}")

In [ ]:
best_wmae, best_cps, best_yo = None, None, None
with mlflow.start_run(run_name="Prophet_CV"):
    for cps in (0.05, 0.5):
        for yo in (10, 20):
            with mlflow.start_run(run_name=f"cps{cps}_yo{yo}", nested=True):
                fc, _ = fit_store_prophet(dev_train, valid_dates,
                                          changepoint_prior_scale=cps, yearly_order=yo)
                score = wmae(dev_valid["Weekly_Sales"],
                             disagg_predict(fc, shares, dev_valid),
                             dev_valid["IsHoliday"])
                mlflow.log_params({"changepoint_prior_scale": cps,
                                   "yearly_fourier_order": yo})
                mlflow.log_metric("valid_wmae", score)
                print(f"cps={cps}, yearly_order={yo} -> WMAE {score:.0f}")
                if best_wmae is None or score < best_wmae:
                    best_wmae, best_cps, best_yo = score, cps, yo

print(f"\nსაუკეთესო: cps={best_cps}, yearly_order={best_yo} (WMAE {best_wmae:.0f})")

In [ ]:
class ShareDisaggPyfunc(mlflow.pyfunc.PythonModel):
    """Pipeline-კონტრაქტი: raw test რიგები -> პროგნოზი.
    შიგნით ინახება store-დონის პროგნოზების ცხრილი + dept-წილები."""
    def __init__(self, store_fc, shares):
        self.store_fc, self.shares = store_fc, shares
    def predict(self, context, model_input, params=None):
        X = model_input.copy()
        X["Date"] = pd.to_datetime(X["Date"])
        m = (X.merge(self.store_fc, on=["Store", "Date"], how="left")
              .merge(self.shares, on=["Store", "Dept"], how="left"))
        return (m["store_pred"] * m["share"]).fillna(0.0).to_numpy()

test_dates = pd.date_range(test["Date"].min(), test["Date"].max(), freq="W-FRI")

with mlflow.start_run(run_name="Prophet_Final") as run:
    store_fc_full, _ = fit_store_prophet(train, test_dates,
                                         changepoint_prior_scale=best_cps,
                                         yearly_order=best_yo)
    pyfunc_model = ShareDisaggPyfunc(store_fc_full, dept_shares(train))
    mlflow.log_params({"changepoint_prior_scale": best_cps,
                       "yearly_fourier_order": best_yo})
    mlflow.log_metric("valid_wmae", best_wmae)
    mlflow.pyfunc.log_model(name="model", python_model=pyfunc_model)
    prophet_final_run_id = run.info.run_id

print("run:", prophet_final_run_id)

In [ ]:
REGISTER_AS_CHAMPION = False

if REGISTER_AS_CHAMPION:
    from mlflow import MlflowClient
    mv = mlflow.register_model(f"runs:/{prophet_final_run_id}/model", REGISTERED_MODEL_NAME)
    MlflowClient().set_registered_model_alias(REGISTERED_MODEL_NAME, "champion", mv.version)
    print(f"დარეგისტრირდა: {REGISTERED_MODEL_NAME} v{mv.version} (alias: champion)")